**Import Libraries**

In [1]:
import pandas as pd
import re
import random

**Load Dataset**

In [2]:
df = pd.read_csv("orders_data.csv")

print("Dataset Loaded Successfully ✅")
print(df.head())

Dataset Loaded Successfully ✅
     OrderID                 Date CustomerID  Product  Quantity  UnitPrice  \
0  ORD200000  2023-01-04 00:00:00     C72649  Monitor         5     570.62   
1  ORD200001  2024-08-23 00:00:00     C75739    Phone         2     151.35   
2  ORD200002  2024-02-27 00:00:00     C81728   Tablet         5     550.68   
3  ORD200003  2023-10-15 00:00:00     C33540    Chair         1     273.19   
4  ORD200004  2025-05-08 00:00:00     C81840  Printer         4     626.01   

  ShippingAddress PaymentMethod OrderStatus TrackingNumber  ItemsInCart  \
0     928 Main St    Debit Card     Shipped    TRK37947903            7   
1     823 Main St        Online     Shipped    TRK91186779            3   
2     512 Main St   Credit Card   Cancelled    TRK42903982            8   
3     275 Main St    Debit Card    Returned    TRK62788070            5   
4     668 Main St        Online   Delivered    TRK29241424            8   

  CouponCode ReferralSource  TotalPrice  
0     SA

**Knowledge Base**

In [3]:
# -------------------------------
# KNOWLEDGE BASE
# -------------------------------

RESPONSES = {

    "hello": [
        "👋 Hello! Welcome to IntelliCart AI.",
        "Hi! How can I help you today?"
    ],

    "hi": [
        "Hello!",
        "Hi there!"
    ],
    "hey": [
        "Hello!",
        "Hi there!"
    ],
    "howdy": [
        "Hello!",
        "Hi there!"
    ],
    "hii": [
        "Hello!",
        "Hi there!"
    ],

    "how are you": [
        "I'm doing great! Thanks for asking 😊"
    ],

    "what is your name": [
        "I'm IntelliCart AI, a Rule-Based Shopping Assistant."
    ],

    "help": [
"""

I can help you with:

• show products
• search <product>
• track <order_id>
• show delivered orders
• show shipped orders
• show cancelled orders
• payment <payment_method>
• coupon <coupon_code>
• referral <source>
• total revenue
• total orders
• average price
• highest price
• lowest price
• contact
• bye

"""
    ],

    "contact": [
        "📧 support@intellicart.ai\n📞 +91-9876543210"
    ],

    "thanks": [
        "You're welcome 😊",
        "Happy to help!"
    ],

    "thank you": [
        "You're welcome!"
    ]
}

EXIT_COMMANDS = {

    "bye",

    "exit",

    "quit",

    "goodbye"

}

DEFAULT_RESPONSE = "❌ Sorry! I didn't understand. Type 'help' for available commands."

**Helper Functions**

In [4]:
def clean_text(text):
    """
    Cleans user input.
    """

    return text.lower().strip()


def random_reply(intent):
    """
    Returns a random response.
    """

    return random.choice(RESPONSES[intent])

**Product Functions**

In [5]:
def show_products():

    products = df["Product"].unique()

    return "\n".join(products)

def search_product(product):

    result = df[
        df["Product"].str.lower() == product.lower()
    ]

    if result.empty:
        return "❌ Product not found."

    row = result.iloc[0]

    return f"""
📦 Product Details

Product : {row['Product']}

Unit Price : ₹{row['UnitPrice']}

Quantity : {row['Quantity']}

Items In Cart : {row['ItemsInCart']}
"""


def track_order(order_id):

    result = df[
        df["OrderID"].str.upper() == order_id.upper()
    ]

    if result.empty:
        return "❌ Order not found."

    row = result.iloc[0]

    return f"""
📦 Order Details

Order ID : {row['OrderID']}

Product : {row['Product']}

Status : {row['OrderStatus']}

Tracking Number : {row['TrackingNumber']}

Payment Method : {row['PaymentMethod']}

Total Price : ₹{row['TotalPrice']}
"""
def show_products():

    products = df["Product"].unique()

    return "\n".join(products)

def orders_by_status(status):

    result = df[df["OrderStatus"].str.lower() == status.lower()]

    if result.empty:
        return "No orders found."

    output = f"\n📦 {status} Orders\n"
    output += "-" * 50 + "\n"

    for _, row in result.iterrows():
        output += (
            f"Order ID : {row['OrderID']}\n"
            f"Product  : {row['Product']}\n"
            f"Price    : ₹{row['TotalPrice']}\n"
            f"Status   : {row['OrderStatus']}\n"
            + "-" * 50 + "\n"
        )

    return output
def payment_method(method):

    result = df[df["PaymentMethod"].str.lower() == method.lower()]

    if result.empty:
        return "No records found."

    output = f"\n💳 Orders paid using {method.title()}\n"
    output += "-" * 50 + "\n"

    for _, row in result.iterrows():
        output += (
            f"Order ID : {row['OrderID']}\n"
            f"Product  : {row['Product']}\n"
            f"Price    : ₹{row['TotalPrice']}\n"
            + "-" * 50 + "\n"
        )

    return output
def coupon_search(code):

    result = df[df["CouponCode"].str.lower() == code.lower()]

    if result.empty:
        return "Coupon not found."

    output = f"\n🏷 Orders using {code.upper()}\n"
    output += "-" * 50 + "\n"

    for _, row in result.iterrows():
        output += (
            f"Order ID : {row['OrderID']}\n"
            f"Product  : {row['Product']}\n"
            f"Discount : {row['CouponCode']}\n"
            + "-" * 50 + "\n"
        )

    return output

def referral_source(source):

    result = df[
        df["ReferralSource"].str.lower() == source.lower()
    ]

    if result.empty:
        return "No records found."

    return result[
        [
            "OrderID",
            "Product",
            "ReferralSource"
        ]
    ]

def total_revenue():

    revenue = df["TotalPrice"].sum()

    return f"💰 Total Revenue : ₹{revenue:.2f}"

def average_price():

    avg = df["TotalPrice"].mean()

    return f"Average Order Value : ₹{avg:.2f}"

def highest_price():

    row = df.loc[
        df["TotalPrice"].idxmax()
    ]

    return f"""
Highest Order

Order ID : {row['OrderID']}

Product : {row['Product']}

Total Price : ₹{row['TotalPrice']}
"""

def lowest_price():

    row = df.loc[
        df["TotalPrice"].idxmin()
    ]

    return f"""
Lowest Order

Order ID : {row['OrderID']}

Product : {row['Product']}

Total Price : ₹{row['TotalPrice']}
"""

def total_orders():

    return f"Total Orders : {len(df)}"

**Response Engine**

In [6]:
def chatbot(user):

    user = clean_text(user)

    if user in EXIT_COMMANDS:
        return "exit"

    if user in RESPONSES:
        return random_reply(user)

    if user == "show products":
        return show_products()

    if user.startswith("search"):
        product = user.replace("search", "").strip()
        return search_product(product)

    if user.startswith("track"):
        order = user.replace("track", "").strip()
        return track_order(order)

    if user == "show delivered orders":
        return orders_by_status("Delivered")

    if user == "show shipped orders":
        return orders_by_status("Shipped")

    if user == "show cancelled orders":
        return orders_by_status("Cancelled")

    if user.startswith("payment"):
        method = user.replace("payment", "").strip()
        return payment_method(method)

    if user.startswith("coupon"):
        code = user.replace("coupon", "").strip()
        return coupon_search(code)

    if user.startswith("referral"):
        source = user.replace("referral", "").strip()
        return referral_source(source)

    if user == "highest price":
        return highest_price()

    if user == "lowest price":
        return lowest_price()

    if user == "average price":
        return average_price()

    if user == "total revenue":
        return total_revenue()

    if user == "total orders":
        return total_orders()

    return DEFAULT_RESPONSE

**Main Chatbot**

In [ ]:
print("="*60)
print("🛒 IntelliCart AI")
print("Rule-Based Conversational Assistant")
print("="*60)

print("""
Available Commands

👋 hello

📦 search monitor

📦 show products

🚚 track ORD200001

🚚 show delivered orders

💳 payment debit card

🎟 coupon SAVE10

📈 total revenue

📈 highest price

👋 bye
""")

while True:

    user = input("\nYou : ")

    response = chatbot(user)

    if response == "exit":
        print("\n🤖 IntelliCart AI")
        print("-" * 50)
        print("Goodbye! 👋")
        print("-" * 50)
        break

    print("\n🤖 IntelliCart AI")
    print("-" * 50)
    print(response)
    print("-" * 50)

🛒 IntelliCart AI
Rule-Based Conversational Assistant

Available Commands

👋 hello

📦 search monitor

📦 show products

🚚 track ORD200001

🚚 show delivered orders

💳 payment debit card

🎟 coupon SAVE10

📈 total revenue

📈 highest price

👋 bye


🤖 IntelliCart AI
--------------------------------------------------
Hi there!
--------------------------------------------------

🤖 IntelliCart AI
--------------------------------------------------

📦 Product Details

Product : Monitor

Unit Price : ₹570.62

Quantity : 5

Items In Cart : 7

--------------------------------------------------
